# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Dataset Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display metadata summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s. These IDs are used for referring to record sets and data columns throughout this exploration.

Let's first list all record sets and display their relevant information (using `@id`).

In [ ]:
# List all record sets using their @id
print("Available record sets (@id):")
for rs in dataset.record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', '(no name)')})")

# For demonstration, let's display first 3 records from each record set
print('\nSample records from each record set:\n')
for rs in dataset.record_sets:
    record_set_id = rs['@id']
    print(f"\nRecord set {record_set_id}:")
    try:
        records_iter = dataset.records(record_set=record_set_id)
        for i, record in enumerate(records_iter):
            if i >= 3:
                break
            print(record)
    except Exception as e:
        print(f"  [Error loading records: {str(e)}]")

# Also, for one record set, list available field @id's
if dataset.record_sets:
    rs0 = dataset.record_sets[0]
    record_set_id = rs0['@id']
    print(f"\nFields in record set {record_set_id} (@id):")
    for field in rs0.get('field', []):
        if isinstance(field, dict) and '@id' in field:
            print(f"  - {field['@id']}")
        elif isinstance(field, str):
            print(f"  - {field}")


## 3. Data Extraction
Load data from each record set into a pandas DataFrame. All references to record sets and fields use the respective `@id` strings.

In [ ]:
# Extract data from each record set using their @id
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

if dataframes:
    # Select one record set to demonstrate
    example_rs_id = next(iter(dataframes.keys()))
    print(f"Record set '{example_rs_id}' columns:")
    print(dataframes[example_rs_id].columns.tolist())
    print("\nSample rows:")
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Here, we'll apply some processing steps:
- Filter based on a numeric column.
- Normalize values.
- Group and aggregate by a categorical field.

Again, all columns are referenced by their `@id`.

In [ ]:
# Choose an example record set and numeric/categorical fields to demonstrate
if dataframes:
    df = dataframes[example_rs_id]

    # Try to identify numeric (@id) columns heuristically
    # For the FAIR2 CRC dataset, let's check column names
    print("Available columns:")
    print(df.columns.tolist())

    # As an example, select column '@id' that seems numeric [adjust below if necessary]:
    numeric_candidates = [col for col in df.columns if any(s in col.lower() for s in ['age', 'interval', 'time', 'score', 'duration'])]

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field_id}")
        
        # Remove missing/non-numeric values
        filtered = pd.to_numeric(df[numeric_field_id], errors='coerce').notna()
        dfn = df[filtered].copy()
        dfn[numeric_field_id] = pd.to_numeric(dfn[numeric_field_id], errors='coerce')
        threshold = dfn[numeric_field_id].mean()
        filtered_df = dfn[dfn[numeric_field_id] > threshold]
        print(f"Filtered rows with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows")

        # Normalize the field
        field_norm = f"{numeric_field_id}_normalized"
        filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered rows:")
        print(filtered_df[[numeric_field_id, field_norm]].head())
        
        # Try to group by a categorical field if available
        cat_candidates = [col for col in df.columns if col != numeric_field_id and (df[col].dtype == object or df[col].dtype.name == 'category')]
        if cat_candidates:
            group_field_id = cat_candidates[0]
            print(f"\nGrouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_value')
            print(grouped_df.head())
    else:
        print("No obvious numeric fields found for analysis.")


## 5. Visualization
Let's visualize value distribution for the selected numeric field using matplotlib and seaborn. Adjust the field as needed depending on the dataset columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[example_rs_id]
    # Use the same numeric field as before
    if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
        numvals = pd.to_numeric(df[numeric_field_id], errors='coerce').dropna()
        plt.figure(figsize=(7, 4))
        sns.histplot(numvals, bins=10)
        plt.xlabel(numeric_field_id)
        plt.ylabel('Frequency')
        plt.title(f"Distribution of {numeric_field_id}")
        plt.show()
    else:
        print("No suitable numeric field found for visualization.")

## 6. Conclusion
In this notebook, we've:
- Loaded the FAIR² CRC dataset using the Croissant schema and `mlcroissant`
- Explored structured metadata and available record sets using `@id`
- Loaded and processed records with pandas DataFrames
- Performed exploratory filtering, normalization, and grouping using `@id` fields
- Visualized value distributions for quantitative analysis

This demonstrates a typical reproducible workflow for exploring Croissant-structured FAIR datasets in Python!